# Day 6：模型量化与 vLLM 部署实践 — 从量化到上线

> **目标**：使用 AutoGPTQ 对 7B 模型进行 GPTQ 4-bit 量化；使用 AutoAWQ 进行 AWQ 4-bit 量化；对比量化前后的推理速度、显存占用和生成质量；使用 vLLM 部署量化模型为 OpenAI 兼容 API 服务；进行并发压测和吞吐量分析。

---

## 总体流程

```
Part 1: 环境准备与模型加载
  安装依赖 → 加载 FP16 基座模型 → 基线推理

Part 2: GPTQ 量化
  准备校准数据 → AutoGPTQ 量化 → 保存量化模型

Part 3: AWQ 量化
  AutoAWQ 量化 → 保存量化模型

Part 4: 量化模型推理 Benchmark
  FP16 vs GPTQ vs AWQ → 速度/显存/质量对比

Part 5: vLLM 部署推理服务
  离线推理 → API 服务 → 并发压测
```

In [ ]:
from __future__ import annotations

import gc
import json
import os
import time
from typing import Optional

import torch
import numpy as np

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

---

## Part 1：环境准备与模型加载

### 1.1 安装依赖

```bash
pip install transformers accelerate
pip install auto-gptq
pip install autoawq
pip install vllm
pip install datasets
```

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

MODEL_ID = "meta-llama/Llama-2-7b-hf"
# 如果无法访问 LLaMA-2，可替换为其他 7B 模型:
# MODEL_ID = "Qwen/Qwen2-7B"
# MODEL_ID = "mistralai/Mistral-7B-v0.1"

OUTPUT_DIR_GPTQ = "./llama2-7b-gptq-4bit"
OUTPUT_DIR_AWQ = "./llama2-7b-awq-4bit"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

### 1.2 加载 FP16 模型 (基线)

In [ ]:
def get_gpu_memory_mb():
    """获取当前 GPU 已用显存 (MB)"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0


def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 128,
                  temperature: float = 0.7) -> tuple[str, float]:
    """生成文本并返回 (输出文本, 耗时秒)"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return text, elapsed


# 加载 FP16 模型
print("Loading FP16 model...")
mem_before = get_gpu_memory_mb()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

mem_after = get_gpu_memory_mb()
print(f"FP16 model loaded. GPU memory: {mem_after:.0f} MB (+{mem_after - mem_before:.0f} MB)")

# 基线推理
test_prompt = "Explain the concept of attention mechanism in transformers:"
text, elapsed = generate_text(model_fp16, tokenizer, test_prompt)
n_tokens = len(tokenizer.encode(text))
print(f"\nFP16 Baseline:")
print(f"  Generated {n_tokens} tokens in {elapsed:.2f}s ({n_tokens/elapsed:.1f} tok/s)")
print(f"  Output: {text[:200]}...")

In [ ]:
# 释放 FP16 模型
del model_fp16
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {get_gpu_memory_mb():.0f} MB")

---

## Part 2：GPTQ 量化

### 2.1 准备校准数据

GPTQ 需要校准数据来计算 Hessian 矩阵 $H = XX^T$。

In [ ]:
from datasets import load_dataset

def prepare_calibration_data(tokenizer, n_samples: int = 128,
                              max_length: int = 2048,
                              dataset_name: str = "wikitext",
                              dataset_config: str = "wikitext-2-raw-v1"):
    """准备校准数据集"""
    dataset = load_dataset(dataset_name, dataset_config, split="train")

    calibration_data = []
    for sample in dataset:
        text = sample["text"].strip()
        if len(text) < 100:
            continue
        tokenized = tokenizer(
            text, return_tensors="pt",
            max_length=max_length, truncation=True
        )
        if tokenized["input_ids"].shape[1] >= 128:
            calibration_data.append(tokenized["input_ids"])
        if len(calibration_data) >= n_samples:
            break

    print(f"Prepared {len(calibration_data)} calibration samples")
    return calibration_data


calibration_data = prepare_calibration_data(tokenizer, n_samples=128)

### 2.2 执行 GPTQ 量化

In [ ]:
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

# GPTQ 量化配置
gptq_config = BaseQuantizeConfig(
    bits=4,            # 4-bit 量化
    group_size=128,    # 每 128 个权重共享量化参数
    desc_act=True,     # 按激活值排序列顺序 (精度更高)
    sym=True,          # 对称量化
)

print("Loading model for GPTQ quantization...")
model_gptq = AutoGPTQForCausalLM.from_pretrained(
    MODEL_ID,
    quantize_config=gptq_config,
)

print("Starting GPTQ quantization (this may take 10-30 minutes)...")
start = time.perf_counter()

model_gptq.quantize(
    calibration_data,
    batch_size=1,
)

gptq_time = time.perf_counter() - start
print(f"GPTQ quantization completed in {gptq_time:.1f}s ({gptq_time/60:.1f} min)")

# 保存
model_gptq.save_quantized(OUTPUT_DIR_GPTQ)
tokenizer.save_pretrained(OUTPUT_DIR_GPTQ)
print(f"Saved GPTQ model to {OUTPUT_DIR_GPTQ}")

# 检查模型大小
total_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR_GPTQ, f))
    for f in os.listdir(OUTPUT_DIR_GPTQ)
    if f.endswith((".safetensors", ".bin"))
)
print(f"GPTQ model size: {total_size / 1024**3:.2f} GB")

del model_gptq
gc.collect()
torch.cuda.empty_cache()

---

## Part 3：AWQ 量化

In [ ]:
from awq import AutoAWQForCausalLM

# AWQ 量化配置
awq_config = {
    "zero_point": True,     # 使用零点 (非对称量化)
    "q_group_size": 128,    # 分组大小
    "w_bit": 4,             # 4-bit
}

print("Loading model for AWQ quantization...")
model_awq = AutoAWQForCausalLM.from_pretrained(MODEL_ID)

print("Starting AWQ quantization...")
start = time.perf_counter()

model_awq.quantize(
    tokenizer,
    quant_config=awq_config,
)

awq_time = time.perf_counter() - start
print(f"AWQ quantization completed in {awq_time:.1f}s ({awq_time/60:.1f} min)")

# 保存
model_awq.save_quantized(OUTPUT_DIR_AWQ)
tokenizer.save_pretrained(OUTPUT_DIR_AWQ)
print(f"Saved AWQ model to {OUTPUT_DIR_AWQ}")

# 检查模型大小
total_size = sum(
    os.path.getsize(os.path.join(OUTPUT_DIR_AWQ, f))
    for f in os.listdir(OUTPUT_DIR_AWQ)
    if f.endswith((".safetensors", ".bin"))
)
print(f"AWQ model size: {total_size / 1024**3:.2f} GB")

del model_awq
gc.collect()
torch.cuda.empty_cache()

---

## Part 4：量化模型推理 Benchmark

### 4.1 加载量化模型

In [ ]:
def benchmark_model(model, tokenizer, model_name: str,
                    prompts: list[str], max_new_tokens: int = 128,
                    n_warmup: int = 2):
    """对模型进行推理 benchmark"""
    print(f"\n{'=' * 60}")
    print(f"Benchmark: {model_name}")
    print(f"{'=' * 60}")

    gpu_mem = get_gpu_memory_mb()
    print(f"GPU memory: {gpu_mem:.0f} MB")

    # Warmup
    for _ in range(n_warmup):
        generate_text(model, tokenizer, prompts[0], max_new_tokens=32)

    # Benchmark
    total_tokens = 0
    total_time = 0
    outputs = []

    for prompt in prompts:
        text, elapsed = generate_text(model, tokenizer, prompt, max_new_tokens)
        n_tokens = len(tokenizer.encode(text))
        total_tokens += n_tokens
        total_time += elapsed
        outputs.append(text)
        print(f"  Prompt: {prompt[:50]}...")
        print(f"    {n_tokens} tokens in {elapsed:.2f}s ({n_tokens/elapsed:.1f} tok/s)")

    avg_tps = total_tokens / total_time
    print(f"\n  Average: {avg_tps:.1f} tokens/s")
    print(f"  GPU memory: {gpu_mem:.0f} MB")

    return {
        "model_name": model_name,
        "gpu_memory_mb": gpu_mem,
        "avg_tokens_per_sec": avg_tps,
        "total_tokens": total_tokens,
        "total_time": total_time,
        "outputs": outputs,
    }


# 测试 prompts
test_prompts = [
    "Explain the difference between GPTQ and AWQ quantization methods:",
    "Write a Python function to implement binary search:",
    "What are the key innovations in the Transformer architecture?",
    "Describe the attention mechanism step by step:",
]

In [ ]:
results = []

# 1. FP16 基线
print("Loading FP16 model...")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
)
result_fp16 = benchmark_model(model_fp16, tokenizer, "FP16", test_prompts)
results.append(result_fp16)
del model_fp16; gc.collect(); torch.cuda.empty_cache()

# 2. GPTQ 量化模型
if os.path.exists(OUTPUT_DIR_GPTQ):
    print("\nLoading GPTQ model...")
    model_gptq = AutoGPTQForCausalLM.from_quantized(
        OUTPUT_DIR_GPTQ, device="cuda:0",
    )
    result_gptq = benchmark_model(model_gptq, tokenizer, "GPTQ-4bit", test_prompts)
    results.append(result_gptq)
    del model_gptq; gc.collect(); torch.cuda.empty_cache()

# 3. AWQ 量化模型
if os.path.exists(OUTPUT_DIR_AWQ):
    print("\nLoading AWQ model...")
    model_awq = AutoAWQForCausalLM.from_quantized(
        OUTPUT_DIR_AWQ, fuse_layers=True,
    )
    result_awq = benchmark_model(
        model_awq.model, tokenizer, "AWQ-4bit", test_prompts,
    )
    results.append(result_awq)
    del model_awq; gc.collect(); torch.cuda.empty_cache()

### 4.2 对比汇总

In [ ]:
print("\n" + "=" * 70)
print("Benchmark 汇总")
print("=" * 70)
print(f"{'Model':<15} {'GPU Mem (MB)':>12} {'Tokens/s':>10} {'vs FP16':>8}")
print("-" * 50)

fp16_tps = results[0]["avg_tokens_per_sec"] if results else 1

for r in results:
    speedup = r["avg_tokens_per_sec"] / fp16_tps
    print(
        f"{r['model_name']:<15} "
        f"{r['gpu_memory_mb']:>10.0f}  "
        f"{r['avg_tokens_per_sec']:>9.1f} "
        f"{speedup:>7.2f}×"
    )

print("\n预期结论:")
print("  - 量化模型显存约为 FP16 的 1/3~1/4")
print("  - 量化模型推理速度提升 1.5~3×")
print("  - AWQ 通常略快于 GPTQ (有专门优化的推理核)")

### 4.3 生成质量对比

In [ ]:
print("\n" + "=" * 70)
print("生成质量对比 (第一个 prompt)")
print("=" * 70)

for r in results:
    print(f"\n--- {r['model_name']} ---")
    print(r["outputs"][0][:300])
    print("...")

---

## Part 5：vLLM 部署推理服务

### 5.1 vLLM 离线推理

vLLM 使用 PagedAttention 和 Continuous Batching，
在高并发场景下吞吐量远超 HuggingFace Transformers。

In [ ]:
from vllm import LLM, SamplingParams

# 加载量化模型 (AWQ)
print("Loading AWQ model with vLLM...")
llm = LLM(
    model=OUTPUT_DIR_AWQ,
    quantization="awq",
    dtype="half",
    gpu_memory_utilization=0.9,
    max_model_len=4096,
)

# 采样配置
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256,
)

# 批量推理
prompts = [
    "What is machine learning?",
    "Explain PagedAttention in simple terms.",
    "Write a Python function to compute Fibonacci numbers.",
    "What are the benefits of model quantization?",
]

start = time.perf_counter()
outputs = llm.generate(prompts, sampling_params)
elapsed = time.perf_counter() - start

total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
print(f"\nvLLM batch inference: {len(prompts)} prompts")
print(f"  Total tokens: {total_tokens}")
print(f"  Time: {elapsed:.2f}s")
print(f"  Throughput: {total_tokens / elapsed:.1f} tokens/s")

for i, output in enumerate(outputs):
    print(f"\nPrompt {i+1}: {prompts[i]}")
    print(f"Output: {output.outputs[0].text[:200]}...")

### 5.2 吞吐量对比: vLLM vs HuggingFace

In [ ]:
def benchmark_vllm_throughput(llm, n_prompts: int = 32,
                               max_tokens: int = 128):
    """测试 vLLM 在不同并发量下的吞吐量"""
    prompts = [f"Question {i}: Explain concept number {i} in machine learning."
               for i in range(n_prompts)]

    params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=max_tokens)

    # 批量大小测试
    batch_sizes = [1, 4, 8, 16, 32]
    results = []

    for bs in batch_sizes:
        if bs > n_prompts:
            break

        batch_prompts = prompts[:bs]

        # Warmup
        _ = llm.generate(batch_prompts[:1], params)

        start = time.perf_counter()
        outputs = llm.generate(batch_prompts, params)
        elapsed = time.perf_counter() - start

        total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        tps = total_tokens / elapsed

        results.append({
            "batch_size": bs,
            "total_tokens": total_tokens,
            "elapsed": elapsed,
            "tokens_per_sec": tps,
        })

        print(f"  Batch={bs:3d}: {total_tokens:5d} tokens in {elapsed:6.2f}s "
              f"→ {tps:7.1f} tok/s")

    return results


print("vLLM 吞吐量测试")
print("-" * 60)
vllm_results = benchmark_vllm_throughput(llm)

### 5.3 启动 OpenAI 兼容 API 服务

vLLM 支持 OpenAI 兼容的 API 接口：

```bash
# 在终端中运行
python -m vllm.entrypoints.openai.api_server \
    --model ./llama2-7b-awq-4bit \
    --quantization awq \
    --dtype half \
    --port 8000 \
    --gpu-memory-utilization 0.9 \
    --max-model-len 4096
```

In [ ]:
# 如果 vLLM API 服务已启动，可以用以下代码测试

import requests

VLLM_API_URL = "http://localhost:8000"


def test_vllm_api(prompt: str, max_tokens: int = 128):
    """测试 vLLM OpenAI 兼容 API"""
    try:
        response = requests.post(
            f"{VLLM_API_URL}/v1/completions",
            json={
                "model": OUTPUT_DIR_AWQ,
                "prompt": prompt,
                "max_tokens": max_tokens,
                "temperature": 0.7,
            },
            timeout=60,
        )
        result = response.json()
        return result["choices"][0]["text"]
    except requests.ConnectionError:
        return "[vLLM API 服务未启动 — 请先在终端运行上方的启动命令]"


# 测试 API
api_result = test_vllm_api("What is FlashAttention?")
print(f"API Response: {api_result[:300]}")

### 5.4 并发压测脚本

In [ ]:
import concurrent.futures


def concurrent_stress_test(
    n_requests: int = 20,
    n_concurrent: int = 5,
    max_tokens: int = 64,
):
    """并发压测 vLLM API"""
    prompts = [
        f"Question {i}: Explain concept {i} in deep learning in 2-3 sentences."
        for i in range(n_requests)
    ]

    results = []

    def send_request(prompt):
        start = time.perf_counter()
        try:
            response = requests.post(
                f"{VLLM_API_URL}/v1/completions",
                json={
                    "model": OUTPUT_DIR_AWQ,
                    "prompt": prompt,
                    "max_tokens": max_tokens,
                    "temperature": 0.7,
                },
                timeout=120,
            )
            elapsed = time.perf_counter() - start
            result = response.json()
            n_tokens = result["usage"]["completion_tokens"]
            return {"success": True, "elapsed": elapsed, "n_tokens": n_tokens}
        except Exception as e:
            elapsed = time.perf_counter() - start
            return {"success": False, "elapsed": elapsed, "error": str(e)}

    print(f"Concurrent stress test: {n_requests} requests, {n_concurrent} concurrent")
    print("-" * 60)

    start_total = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_concurrent) as executor:
        futures = [executor.submit(send_request, p) for p in prompts]
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())
    total_time = time.perf_counter() - start_total

    # 统计
    successful = [r for r in results if r["success"]]
    if successful:
        total_tokens = sum(r["n_tokens"] for r in successful)
        avg_latency = np.mean([r["elapsed"] for r in successful])
        p50 = np.percentile([r["elapsed"] for r in successful], 50)
        p95 = np.percentile([r["elapsed"] for r in successful], 95)
        throughput = total_tokens / total_time

        print(f"  Successful requests: {len(successful)}/{n_requests}")
        print(f"  Total time:   {total_time:.2f}s")
        print(f"  Total tokens: {total_tokens}")
        print(f"  Throughput:   {throughput:.1f} tokens/s")
        print(f"  Avg latency:  {avg_latency:.2f}s")
        print(f"  P50 latency:  {p50:.2f}s")
        print(f"  P95 latency:  {p95:.2f}s")
    else:
        print("  [所有请求失败 — 请确保 vLLM API 服务已启动]")


# 运行压测 (需要先在终端启动 vLLM API 服务)
concurrent_stress_test(n_requests=20, n_concurrent=5)

---

## 总结

### 本日核心收获

| 实践环节 | 掌握技能 |
|---------|----------|
| GPTQ 量化 | AutoGPTQ 配置、校准数据准备、量化执行 |
| AWQ 量化 | AutoAWQ 配置、量化执行 |
| 推理 Benchmark | 显存/速度/质量三维对比 |
| vLLM 部署 | 离线推理、API 服务、并发压测 |

### 关键数据

| 指标 | FP16 | GPTQ-4bit | AWQ-4bit |
|------|------|-----------|----------|
| 模型大小 | ~13 GB | ~3.6 GB | ~3.6 GB |
| GPU 显存 | ~14 GB | ~4 GB | ~4 GB |
| 推理速度 | 基准 | ~2× | ~2.5× |
| PPL 损失 | 0 | +0.1~0.2 | +0.1 |

### 与后续内容的衔接

```
Day 6: 量化实验 + vLLM 部署
  → Day 7: TensorRT-LLM / SGLang 更高级的部署方案
          + Speculative Decoding 进一步加速推理
```